# Grover Adaptive SearchとQUBO

このチュートリアルでは、[Groverのアルゴリズム](../304_qaa_qae_grover_gas.ipynb)と[量子振幅増幅](../304_qaa_qae_grover_gas.ipynb)の一般化版を用いて、一般のQUBO問題を解く量子アルゴリズムを紹介します。これは、ゲート方式の量子コンピューティングを用いたQAOAアルゴリズムに対する(やや複雑な)代替手法であることに注意してください。

断熱量子計算はゲート方式の量子計算と等価であることが知られています。RolandとCerfによる重要な論文[[1]]は、サイズ$N$の解空間全体に対するGrover探索によって得られる$O(\sqrt N)$の高速化が、QAOAアルゴリズムの高速化とちょうど一致することを証明しています。

つまり、ゲート方式のアルゴリズムは理論的にはQAOAと等価ですが、多数の量子ビットからなるコヒーレントな回路を構築する必要があり、近い将来に量子コンピュータ上で実用化するのは難しいかもしれません。

[1]:https://arxiv.org/abs/quant-ph/0107015

In [1]:
# Install blueqat-sdk to get started.
# !pip install git+https://github.com/blueqat/blueqatSDK

## QUBO

QUBO(Quadratic Unconstrained Binary Optimization、制約なし二次二値最適化問題)は、最も一般的な形では次の式を最小化する問題です。
$$X^TQX+RX,$$
ここで、
+ $Q$は$n\times n$の半正定値行列、
+ $R$は$1\times n$の行ベクトル、
+ $X$は$n$次元の$\{0,1\}$値の列ベクトルです。

_注: 制約付き二次最適化問題ではベクトル$X$に追加の制約が課されます。これらの制約が線形である限り、そのような問題はQUBOに帰着できることがよく知られています(参考: [[2],[3]])。_

QUBO問題を高速に解くことは、現代の計算科学において非常に重要な課題です。これは、この章の前のチュートリアルで議論したMax-Cut、Graph Coloring、Vertex Coverなどの多くの最適化問題がQUBOとして定式化できるためです。

[2]:https://arxiv.org/pdf/1912.04088.pdf
[3]:https://arxiv.org/pdf/1811.11538.pdf

## Groverのアルゴリズム vs. Grover Adaptive Search Algorithm(GAS)

このチュートリアルでは、$n=3$の場合について[Gilliam-Woerner-Gonciuleaのアルゴリズム](https://arxiv.org/pdf/1912.04088.pdf)を実装します。ただし、GASについて説明する前に、Grover探索の基本的な考え方をおさらいしておきましょう。

### Groverのアルゴリズム
Grover探索には次の要素が必要です。

+ $\lvert 0\rangle_n$を一様重ね合わせ状態$\lvert\varphi\rangle = \frac1{\sqrt{2^n}}\sum_{i}\lvert i\rangle_n$に写す演算子$A$。実際には$A=H^{\otimes n}$です。
+ 状態の集合$I\subseteq \{0,\ldots,2^n-1\}$に印を付けるオラクル$O$。これは次を満たします。
$$O\lvert\varphi\rangle = \frac1{\sqrt{2^n}}\sum_{i\not\in I}\lvert i\rangle_n - \frac1{\sqrt{2^n}}\sum_{i\in I}\lvert i\rangle_n.$$
+ 状態$\lvert 0\rangle_n$の符号を反転させる拡散演算子$D$。

Grover演算子は$G=ADA^{\dagger}O$であり、これを$A\lvert 0\rangle_n$に対して$\left\lfloor\frac{\pi}{4}\sqrt{\frac{2^n}{|I|}}\right\rfloor$回適用すると、測定結果が高確率で$I$の要素の1つとなる量子状態が得られます。

### Grover Adaptive Search Algorithm
GASを実装するために必要な要素は次の通りです。

+ 次を満たす演算子$A_y$
$$A_y\lvert 0\rangle_n\lvert 0\rangle_m = \frac 1{\sqrt{2^n}}\sum_{x} \lvert x\rangle_n\lvert f(x)-y\rangle_m.$$
+ 次を満たすオラクル$O_y$
$$O_y\lvert x\rangle_n\lvert z\rangle_m = \text{sign}(z)\lvert x\rangle_n\lvert z\rangle_m.$$


**入力:** $f:X\to \mathbb R$、$\lambda >1$、num_iter(デフォルトでは3)。

**アルゴリズム:**
+ $x_1\in X$を一様にサンプリングし、$y_1=f(x_1)$とする、
+ $k=1$、$i=1$とする、
+ while(true):
    - $r_i\in\{0,\ldots,\lceil k-1\rceil\}$を一様ランダムに選ぶ
    - オラクル$A_{y_i}$と$O$を用いて、$r_i$回の反復でGrover探索を適用する。出力を$x,y$とする。
        * もし$y<y_i$なら、$x_{i+1}=x, y_{i+1}=y$として、$k=1$にリセットする。
        * そうでなければ、$x_{i+1}=x_i, y_{i+1}=y_i, k=\lambda k$とする。
    - i++
    - $x_i$の値がnum_iter回変化しなければ終了する。


## 簡単なQUBO問題への実装

次のQUBO問題を考えます。$Q=\begin{pmatrix}1 & 0.5\\ 0.5&2\end{pmatrix}$、$R=\begin{pmatrix} 0 & -3\end{pmatrix}$。このとき、最小化したいのは次の式です。
$$f(x_0,x_1) = x_0^2 + x_0x_1 + 2x_1^2 -3x_1 = x_0 +x_0x_1 -x_1.$$

このプログラムを実装する上での主な難しさは、演算子$A_y$と$O$のための量子回路を構築することです。

2つのレジスタを使用します。
+ 問題の各変数に対応する$n=2$量子ビットの_キーレジスタ_、
+ $f(x_0, x_1)$の様々な値を格納するための$m=2$量子ビットの_値レジスタ_(最初の量子ビットが符号、2番目の量子ビットが値に対応)。

一般のQUBO問題では、$\max\limits_{x\in\{0,1\}^n} |f(x)| < 2^{m-1}$となるように$m$を十分大きく取る必要があります。今回は$m=2$で十分です!

### 演算子$A_y$
これは制御位相演算を用いて実装できます。

+ まず一様重ね合わせ状態を作ります。
+ 固定された$y$に対して、$A_y$は$f(x)-y$を評価する必要があります。これは、まず値レジスタに角度$\frac{-\pi y}{2}$と$-\pi y$の位相ゲートを用いて$-y$を符号化することで実現します。
+ 項$x_0$については、値レジスタの1番目と2番目の量子ビットにそれぞれ角度$\frac{\pi}{2}$と$\pi$の制御位相ゲートを適用します。
+ 項$-x_1$については、角度は$-\frac{\pi}{2}$と$-\pi$になります。
+ 項$x_0x_1$については、2つの多重制御位相ゲートが必要です。
+ 一般に、多項式中の項$ax_{i_1}x_{i_2}\cdots x_{i_k}$に対しては、$i_1,\ldots, i_k$番目の量子ビットを制御とし、$j$番目の値レジスタの量子ビットに角度$\frac{2\pi}{2^m}\cdot a\cdot 2^j$の位相ゲートを適用します。
+ 最後に、これらの制御位相演算の後、値レジスタに逆QFTを適用して値レジスタ上に$\lvert f(x)-y\rangle$を計算します!以下は$A_y$の回路実装です。

![Circuit for A_1](../img/323_gas_ckt1.png)

この回路は2量子ビットゲートを使ってさらに簡略化できます。2重制御位相演算はCPhaseゲートとCNOTゲートに分解できるため、次の回路が得られます。

![Circuit for A_1](../img/323_gas_ckt2.png)

それでは、これをblueqatで実装しましょう。チュートリアルの[QFT回路](../300_qft.ipynb)を思い出してください。

In [2]:
# Import Libraries
from blueqat import Circuit
import math

# Function to apply qft on a list of qubits of circuit
def apply_qft(circuit: Circuit(), qubits):
    num_qubits = len(qubits)
    for i in range(num_qubits):
        circuit.h[qubits[i]]
        for j in range(i+1, num_qubits):
            circuit.cphase(math.pi/(2 ** (j-i)))[qubits[j],qubits[i]] # Apply gate CR_{j-i}(qubit j, qubit i)
    # Reverse the order of qubits at the end
    for i in range(int(num_qubits/2)):
        circuit.swap[qubits[i],qubits[num_qubits-i-1]]

# Function to apply iqft on a list of qubits of circuit
def apply_iqft(circuit: Circuit(), qubits):
    num_qubits = len(qubits)
    # Reverse the order of qubits
    for i in range(int(num_qubits/2)):
        circuit.swap[qubits[i],qubits[num_qubits-i-1]]
    for i in range(num_qubits):
        for j in range(i+1, num_qubits):
            circuit.cphase(-math.pi/(2 ** (j-i)))[qubits[j],qubits[i]] # Apply gate CR_{j-i}(qubit j, qubit i)
        circuit.h[qubits[i]]

In [3]:
# Implementing the operator A_y for our function
def apply_Ay(circuit: Circuit(), yvalue: int):
    # Hadamard transform for Uniform superposition
    for i in range(4):
        circuit.h[i]
    # Encoding -y
    circuit.phase(-math.pi/2 * yvalue)[2]
    circuit.phase(-math.pi * yvalue)[3]
    # Encoding the polynomial
    # monomial: x0
    circuit.cphase(math.pi/2)[0,2]
    circuit.cphase(math.pi)[0,3]
    # monomial: -x1
    circuit.cphase(-math.pi/2)[1,2]
    circuit.cphase(-math.pi)[1,3]
    # monomial: x0*x1
    circuit.cphase(math.pi/4)[1,2]
    circuit.cx[1,0]
    circuit.cphase(-math.pi/4)[0,2]
    circuit.cx[1,0]
    circuit.cphase(math.pi/4)[0,2]
    circuit.cphase(math.pi/2)[1,3]
    circuit.cx[1,0]
    circuit.cphase(-math.pi/2)[0,3]
    circuit.cx[1,0]
    circuit.cphase(math.pi/2)[0,3]
    # IQFT
    apply_iqft(circuit, [2,3])

# A_y dagger:
def apply_Ay_dagger(circuit: Circuit(), yvalue: int):
    # QFT
    apply_qft(circuit, [2,3])
    # Inverting encoded polynomial
    # inverting monomial: x0
    circuit.cphase(-math.pi/2)[0,2]
    circuit.cphase(-math.pi)[0,3]
    # inverting monomial: -x1
    circuit.cphase(math.pi/2)[1,2]
    circuit.cphase(math.pi)[1,3]
    # inverting monomial: x0*x1
    circuit.cphase(-math.pi/4)[1,2]
    circuit.cx[1,0]
    circuit.cphase(math.pi/4)[0,2]
    circuit.cx[1,0]
    circuit.cphase(-math.pi/4)[0,2]
    circuit.cphase(-math.pi/2)[1,3]
    circuit.cx[1,0]
    circuit.cphase(math.pi/2)[0,3]
    circuit.cx[1,0]
    circuit.cphase(-math.pi/2)[0,3]
    # Inverting encoded -y
    circuit.phase(math.pi/2 * yvalue)[2]
    circuit.phase(math.pi * yvalue)[3]
    # Inverting Hadamard transform
    for i in range(4):
        circuit.h[i]

### オラクルO
オラクル$O$は、負の符号を表す値レジスタの最初の量子ビットが$1$である状態の符号を反転させる必要があるため、このオラクルを実装するには$Z$ゲートを適用するだけで十分です。

In [4]:
def apply_oracle_Oy(circuit: Circuit()):
    circuit.z(2)

### Groverの拡散演算D
blueqat-sdk上で$\lvert 0000\rangle$を軸とした反転を実装する必要があります。これは、追加の補助量子ビットを使った次の回路で実現できます。

![Diffusion-ancilla](../img/323_gas_ckt3.png)

In [5]:
def apply_diffusion(circuit: Circuit(), qubits, ancilla):
    if len(qubits) != 4:
        raise ValueError('Length of qubits must be 4')
    circuit.x(qubits)
    circuit.h[qubits[0]]
    circuit.ccx(qubits[2], qubits[3], ancilla)
    circuit.ccx(ancilla, qubits[1], qubits[0])
    circuit.ccx(qubits[2], qubits[3], ancilla)
    circuit.h[qubits[0]]
    circuit.x(qubits)

## メインアルゴリズムの実装

それでは、blueqatのシミュレータ上で$f(x_0,x_1) = x_0+x_0x_1-x_1$を最小化する最適化問題を解いてみましょう!

In [6]:
from random import randint
def evaluate_f(x0, x1):
    return x0 + x0*x1 -x1

# Initialize all variables
x0value = randint(0,1)
x1value = randint(0,1)
yvalue = evaluate_f(x0value, x1value)
print('Initial value: f({},{})={}'.format(x0value, x1value, yvalue))

k = 1
i = 1
lam = 8/7
t = 0

loops_without_improvement = 0

while True:
    rotation_count = randint(0,math.ceil(k-1))
    print('Creating Grover Circuit with {} rotations, k = {}, threshold = {}'.format(rotation_count, k, t))
    circuit = Circuit()
    apply_Ay(circuit, -t)
    for _ in range(rotation_count):
        apply_oracle_Oy(circuit)
        apply_Ay_dagger(circuit, -t)
        apply_diffusion(circuit, range(4), 4)
        apply_Ay(circuit, -t)
    circuit.m[:4]
    # print(circuit)
    result = circuit.run(shots=1000)
    # print(result)
    result_str = result.most_common(1)[0][0]
    # Get the most frequent output (x,y) from the circuit
    # print(result_str)
    x0_curr = int(result_str[0])
    x1_curr = int(result_str[1])
    y_curr = evaluate_f(x0_curr, x1_curr)
    print('Current value: f({},{})={}'.format(x0_curr, x1_curr, y_curr))
    if y_curr < yvalue:
        x0value = x0_curr
        x1value = x1_curr
        yvalue = y_curr
        t = y_curr
        k = 1.0
        circuit.reset()
        loops_without_improvement = 0
    else:
        k = math.ceil(lam * k)
        loops_without_improvement += 1

    print('Loops without improvement: {}'.format(loops_without_improvement))
    if loops_without_improvement == 5:
        break

print('Minimum Value is f({},{})={}'.format(x0value, x1value, yvalue))

Initial value: f(1,0)=1
Creating Grover Circuit with 0 rotations, k = 1, threshold = 0
Current value: f(0,1)=-1
Loops without improvement: 0
Creating Grover Circuit with 0 rotations, k = 1.0, threshold = -1
Current value: f(1,1)=1
Loops without improvement: 1
Creating Grover Circuit with 0 rotations, k = 2, threshold = -1
Current value: f(1,1)=1
Loops without improvement: 2
Creating Grover Circuit with 2 rotations, k = 3, threshold = -1
Current value: f(0,1)=-1
Loops without improvement: 3
Creating Grover Circuit with 3 rotations, k = 4, threshold = -1
Current value: f(1,1)=1
Loops without improvement: 4
Creating Grover Circuit with 0 rotations, k = 5, threshold = -1
Current value: f(1,1)=1
Loops without improvement: 5
Minimum Value is f(0,1)=-1
